In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [20]:
df = pd.read_csv('abalone_original.csv', sep = ',')

df = pd.get_dummies(df, columns=['sex'], drop_first=True)

X = df.drop('rings', axis = 1).values
y = df['rings'].values.reshape(-1,1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [32]:
def relu(x):
  return (x > 0) * x


def relu2deriv(output):
  return output>0

np.random.seed(42)
alpha = 0.001
hidden_size_1 = 5
hidden_size_2 = 4
weights_0_1 = np.random.randn(X_train_scaled.shape[1], hidden_size_1) * 0.1
weights_1_2 = np.random.randn(hidden_size_1, hidden_size_2) * 0.1
weights_2_3 = np.random.randn(hidden_size_2, 1) * 0.1

print("Training Started...")

for iteration in range(100):
  layer_3_error = 0
  for i in range(len(X_train_scaled)):
    layer_0 = X_train_scaled[i:i+1]
    layer_1 = relu(np.dot(layer_0,weights_0_1))
    layer_2 = relu(np.dot(layer_1,weights_1_2))
    layer_3 = np.dot(layer_2,weights_2_3)
    layer_3_error += np.sum((layer_3 - y_train[i:i+1]) ** 2)
    layer_3_delta = (layer_3 - y_train[i:i+1])
    layer_2_delta = np.dot(layer_3_delta, weights_2_3.T)*relu2deriv(layer_2)
    layer_1_delta = np.dot(layer_2_delta, weights_1_2.T)*relu2deriv(layer_1)
    weights_2_3 -= alpha * np.dot(layer_2.T, layer_3_delta)
    weights_1_2 -= alpha * np.dot(layer_1.T, layer_2_delta)
    weights_0_1 -= alpha * np.dot(layer_0.T, layer_1_delta)
  if(iteration % 10 == 9):
    print("Error:" + str(layer_3_error))

print("\nExample Prediction on Test Data:")
layer_0_test = X_test_scaled[0:1]
layer_1_test = relu(np.dot(layer_0_test, weights_0_1))
layer_2_test = relu(np.dot(layer_1_test, weights_1_2))
prediction = np.dot(layer_2_test, weights_2_3)

print(f"Predicted Rings: {prediction[0][0]:.2f}")
print(f"Actual Rings: {y_test[0][0]}")

Training Started...
Error:17157.863177164072
Error:16833.255658938877
Error:16711.214465627083
Error:16665.576857102446
Error:16578.27418636829
Error:16551.662243821705
Error:16554.647184735244
Error:16475.511246574704
Error:16376.731238325663
Error:16313.454944816738

Example Prediction on Test Data:
Predicted Rings: 11.84
Actual Rings: 9


In [30]:
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(hidden_layer_sizes=(5, 4),
                   activation='relu',
                   solver='sgd',
                   learning_rate_init=0.001,
                   max_iter=100,
                   random_state=42)


print("Training Started...")
mlp.fit(X_train_scaled, y_train.ravel())


predictions = mlp.predict(X_test_scaled)


print(f"\nExample Prediction on Test Data:")
print(f"Predicted Rings: {predictions[0]:.2f}")
print(f"Actual Rings: {y_test[0][0]}")

Training Started...

Example Prediction on Test Data:
Predicted Rings: 11.85
Actual Rings: 9


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
